# 🔪 Module 2.2: Edge Detection — Finding Boundaries with Math

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_02_Image_Processing_Basics/02_Edge_Detection/edge_detection.ipynb)

---

## 🎯 Learning Objectives
1. Gradient-based edge detection (Sobel, Prewitt)
2. Canny edge detector — complete algorithm
3. Laplacian and second-order methods
4. Edge detection as RL reward signal

---

In [ ]:
!pip install numpy matplotlib opencv-python-headless scikit-image -q

import numpy as np
import matplotlib.pyplot as plt
import cv2

print("✅ Ready!")

In [ ]:
# Download REAL open-source dataset
from skimage import data
import torchvision

# CIFAR-10: 60,000 real photographs in 10 classes (auto-downloads ~170MB)
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
print(f"✅ CIFAR-10 loaded: {len(cifar10)} real photos (32x32x3)")
print(f"   Classes: {cifar10.classes}")

# scikit-image real test images
from skimage import data
sample_images = {
    'astronaut': data.astronaut(),
    'camera': data.camera(), 
    'coins': data.coins(),
    'horse': data.horse(),
}
print(f"✅ scikit-image: {len(sample_images)} real test images loaded")

## Deep Mathematical Derivation: Edge Detection Operators

### Step 1: Image Gradient (Foundation of All Edge Detection)
An edge is a rapid change in intensity. The gradient of image $I(x,y)$:
$$\nabla I = \left(\frac{\partial I}{\partial x}, \frac{\partial I}{\partial y}\right)$$

**Gradient magnitude** (edge strength): $|\nabla I| = \sqrt{\left(\frac{\partial I}{\partial x}\right)^2 + \left(\frac{\partial I}{\partial y}\right)^2}$

**Gradient direction** (edge orientation): $\theta = \arctan\left(\frac{\partial I / \partial y}{\partial I / \partial x}\right)$

### Step 2: Discrete Derivative Approximation
**Forward difference:** $\frac{\partial I}{\partial x} \approx I[x+1, y] - I[x, y]$

**Central difference (more accurate):** $\frac{\partial I}{\partial x} \approx \frac{I[x+1, y] - I[x-1, y]}{2}$

**Proof of accuracy via Taylor expansion:**
$$I[x+1,y] = I[x,y] + \frac{\partial I}{\partial x} + \frac{1}{2}\frac{\partial^2 I}{\partial x^2} + O(h^3)$$
$$I[x-1,y] = I[x,y] - \frac{\partial I}{\partial x} + \frac{1}{2}\frac{\partial^2 I}{\partial x^2} - O(h^3)$$

Subtracting: $I[x+1,y] - I[x-1,y] = 2\frac{\partial I}{\partial x} + O(h^3)$

Forward difference error: $O(h)$. Central difference error: $O(h^2)$. $\blacksquare$

### Step 3: Sobel Operator Derivation
The Sobel operator combines smoothing (Gaussian-like) with differentiation:

$$G_x = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix} = \begin{bmatrix} 1 \\ 2 \\ 1 \end{bmatrix} \begin{bmatrix} -1 & 0 & 1 \end{bmatrix}$$

**Proof of separability:** The Sobel kernel is the outer product of a smoothing vector $[1, 2, 1]^T$ (approximation to Gaussian) and a differentiation vector $[-1, 0, 1]$.

**Why the weights [1, 2, 1]?**
This is the discrete approximation to $G(\sigma=0.85)$: it averages along the direction perpendicular to the derivative, reducing noise sensitivity. $\blacksquare$

### Step 4: Canny Edge Detector — Complete Mathematical Pipeline

**Step 4a: Gaussian smoothing** — suppress noise:
$$I_{\text{smooth}} = I * G(\sigma)$$

**Step 4b: Gradient computation** — find edge candidates:
$$G_x = I_{\text{smooth}} * K_x, \quad G_y = I_{\text{smooth}} * K_y$$
$$M(x,y) = \sqrt{G_x^2 + G_y^2}, \quad \theta(x,y) = \arctan(G_y / G_x)$$

**Step 4c: Non-maximum suppression** — thin edges to 1-pixel width:
$$M_{\text{thin}}(x,y) = \begin{cases} M(x,y) & \text{if } M(x,y) \geq M(x + \cos\theta, y + \sin\theta) \text{ AND} \\ & \quad M(x,y) \geq M(x - \cos\theta, y - \sin\theta) \\ 0 & \text{otherwise} \end{cases}$$

**Proof of correctness:** Along the gradient direction, a true edge is a local maximum of $M$. Non-edge points along the gradient ray have lower magnitude, so they're suppressed. $\blacksquare$

**Step 4d: Hysteresis thresholding** — connect edges:
- If $M_{\text{thin}}(x,y) > T_{\text{high}}$: **strong edge** (definitely an edge)
- If $T_{\text{low}} < M_{\text{thin}}(x,y) \leq T_{\text{high}}$: **weak edge** (edge only if connected to strong edge)
- If $M_{\text{thin}}(x,y) \leq T_{\text{low}}$: **not an edge**

### Step 5: Laplacian of Gaussian (LoG) — Second-Order Method
**Theorem:** Zero crossings of $\nabla^2(G * I)$ locate edges.

$$\nabla^2 G(x,y;\sigma) = \frac{x^2 + y^2 - 2\sigma^2}{2\pi\sigma^6} \exp\left(-\frac{x^2+y^2}{2\sigma^2}\right)$$

**Proof:** $\nabla^2(G * I) = (\nabla^2 G) * I$ by linearity of convolution and differentiation.

At an edge, $I$ has an inflection point → second derivative crosses zero → $\nabla^2(G*I) = 0$. $\blacksquare$

### RL Connection: Edge Maps as Reward Signals
For an RL agent performing image enhancement, edge preservation serves as a reward component:
$$R_{\text{edge}} = \text{SSIM}(\text{edges}(I_{\text{output}}), \text{edges}(I_{\text{target}}))$$

The agent learns to enhance images while preserving structural edges — a key quality metric.

## Gradient Derivation from Taylor Expansion — First Principles

Edge detection is fundamentally about estimating derivatives of a discrete signal. Here we derive the mathematical foundations rigorously.

### Step 1: First-Order Taylor Expansion in 2D

For a smooth image function $I(x, y)$, the Taylor expansion about point $(x_0, y_0)$:

$$I(x_0 + h, y_0 + k) = I(x_0, y_0) + h \frac{\partial I}{\partial x} + k \frac{\partial I}{\partial y} + \frac{1}{2}\left(h^2 \frac{\partial^2 I}{\partial x^2} + 2hk \frac{\partial^2 I}{\partial x \partial y} + k^2 \frac{\partial^2 I}{\partial y^2}\right) + O(h^3 + k^3)$$

### Step 2: Derivation of Finite Difference Operators

**Forward difference** (set $h=1, k=0$):
$$\frac{\partial I}{\partial x} \approx I(x+1, y) - I(x, y) + O(h)$$

**Central difference** (subtract $I(x-1,y)$ from $I(x+1,y)$):
$$I(x+1,y) = I + I_x + \frac{1}{2}I_{xx} + \frac{1}{6}I_{xxx} + O(h^4)$$
$$I(x-1,y) = I - I_x + \frac{1}{2}I_{xx} - \frac{1}{6}I_{xxx} + O(h^4)$$

Subtracting: $I(x+1,y) - I(x-1,y) = 2I_x + O(h^3)$

$$\therefore \frac{\partial I}{\partial x} = \frac{I(x+1,y) - I(x-1,y)}{2} + O(h^2) \quad \blacksquare$$

The central difference has error $O(h^2)$ — one order better than forward difference $O(h)$.

### Step 3: Second-Order Derivative (Laplacian) from Taylor

Adding instead of subtracting:
$$I(x+1,y) + I(x-1,y) = 2I + I_{xx} + O(h^4)$$

$$\therefore \frac{\partial^2 I}{\partial x^2} = I(x+1,y) - 2I(x,y) + I(x-1,y) + O(h^2)$$

The discrete Laplacian combines both directions:
$$\nabla^2 I = \frac{\partial^2 I}{\partial x^2} + \frac{\partial^2 I}{\partial y^2} = I(x+1,y) + I(x-1,y) + I(x,y+1) + I(x,y-1) - 4I(x,y)$$

This gives the Laplacian kernel:
$$K_{\nabla^2} = \begin{bmatrix} 0 & 1 & 0 \\ 1 & -4 & 1 \\ 0 & 1 & 0 \end{bmatrix}$$

### Step 4: Gradient Magnitude and the Structure Tensor

The gradient vector $\nabla I = (I_x, I_y)^T$ captures local edge information. The **structure tensor** (second-moment matrix) aggregates gradient information over a neighborhood $\mathcal{N}$:

$$\mathbf{J} = \sum_{(x,y) \in \mathcal{N}} w(x,y) \begin{bmatrix} I_x^2 & I_x I_y \\ I_x I_y & I_y^2 \end{bmatrix}$$

The eigenvalues $\lambda_1, \lambda_2$ of $\mathbf{J}$ classify local structure:
- $\lambda_1 \approx \lambda_2 \approx 0$: flat region (no edge)
- $\lambda_1 \gg \lambda_2 \approx 0$: edge (one dominant direction)
- $\lambda_1 \approx \lambda_2 \gg 0$: corner (edges in multiple directions)

This eigenvalue analysis is the mathematical basis of the Harris corner detector and connects edge detection to the broader theory of local image structure.

## Sobel Operator — Derivation as Separable Filter

The Sobel operator is not an arbitrary edge detector — it is the optimal combination of noise suppression and derivative estimation for discrete images.

### Step 1: Design Goals

An edge detection kernel should simultaneously:
1. **Differentiate** in one direction (detect intensity changes)
2. **Smooth** in the perpendicular direction (reduce noise sensitivity)

### Step 2: Separability Derivation

The Sobel-$x$ kernel is the outer product of a smoothing vector and a differentiation vector:

$$\mathbf{G}_x = \mathbf{s} \cdot \mathbf{d}^T = \begin{bmatrix} 1 \\ 2 \\ 1 \end{bmatrix} \begin{bmatrix} -1 & 0 & 1 \end{bmatrix} = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix}$$

**The differentiation vector** $\mathbf{d} = [-1, 0, 1]$ implements the central difference: $\frac{\partial I}{\partial x} \approx \frac{I[x+1] - I[x-1]}{2}$ (ignoring the factor of 2).

**The smoothing vector** $\mathbf{s} = [1, 2, 1]^T$ approximates Gaussian smoothing. It is the binomial coefficient vector $\binom{2}{k}$, which is the discrete approximation to a Gaussian with $\sigma = \sqrt{2}/2 \approx 0.71$.

### Step 3: Why [1, 2, 1] and Not [1, 1, 1]?

**Proof that $[1,2,1]$ is a better Gaussian approximation:**

The ideal discrete Gaussian with $\sigma$: $g[n] \propto e^{-n^2/2\sigma^2}$ for $n \in \{-1, 0, 1\}$.

For $\sigma = 0.71$: $g[-1] = g[1] \propto e^{-1/(2 \cdot 0.5)} = e^{-1} \approx 0.368$, $g[0] \propto 1$.

Normalized: $[0.368, 1, 0.368] / 1.736 \approx [0.212, 0.576, 0.212]$.

Compare: $[1,2,1]/4 = [0.25, 0.5, 0.25]$ — close match!

The box filter $[1,1,1]/3 = [0.333, 0.333, 0.333]$ is a poor Gaussian approximation.

### Step 4: Noise Sensitivity Analysis

**Theorem:** The Sobel operator has lower noise sensitivity than the Prewitt operator.

The Prewitt operator uses $\mathbf{s} = [1, 1, 1]^T$ (box smoothing):
$$\mathbf{G}_x^{\text{Prewitt}} = \begin{bmatrix} -1 & 0 & 1 \\ -1 & 0 & 1 \\ -1 & 0 & 1 \end{bmatrix}$$

For Gaussian noise $\eta \sim \mathcal{N}(0, \sigma^2)$, the variance of the gradient estimate is:
$$\text{Var}[\hat{G}_x] = \sigma^2 \sum_{i,j} K_{ij}^2$$

- **Prewitt:** $\sum K_{ij}^2 = 6 \times 1^2 = 6$
- **Sobel:** $\sum K_{ij}^2 = 4 \times 1^2 + 2 \times 2^2 = 12$, but output magnitude is $4\times$ Prewitt's

SNR comparison: $\text{SNR}_{\text{Sobel}} / \text{SNR}_{\text{Prewitt}} = \frac{4/\sqrt{12}}{2/\sqrt{6}} = \frac{4\sqrt{6}}{2\sqrt{12}} = \sqrt{2} \approx 1.41$

**Result:** Sobel achieves ~41% better SNR than Prewitt. $\blacksquare$

### Step 5: Scharr Operator — Optimized Sobel

The Scharr operator minimizes the angular error of gradient estimation:

$$\mathbf{G}_x^{\text{Scharr}} = \begin{bmatrix} -3 & 0 & 3 \\ -10 & 0 & 10 \\ -3 & 0 & 3 \end{bmatrix} = \begin{bmatrix} 3 \\ 10 \\ 3 \end{bmatrix} \begin{bmatrix} -1 & 0 & 1 \end{bmatrix}$$

The weights $[3, 10, 3]$ are optimized to minimize the rotational asymmetry of the gradient estimate across all orientations — making the Scharr operator more isotropic than Sobel.

## Prewitt, Roberts, and Scharr Operators — A Comparative Mathematical Analysis

Different edge detection kernels make different tradeoffs between accuracy, noise sensitivity, and computational cost.

### Step 1: Roberts Cross Operator (1963)

The simplest gradient operator uses $2 \times 2$ kernels with diagonal differences:

$$K_1 = \begin{bmatrix} 1 & 0 \\ 0 & -1 \end{bmatrix}, \quad K_2 = \begin{bmatrix} 0 & 1 \\ -1 & 0 \end{bmatrix}$$

**Gradient magnitude:** $|\nabla I| = \sqrt{(I * K_1)^2 + (I * K_2)^2}$

**Pros:** Minimum computation (4 multiplications per pixel)
**Cons:** No smoothing → very noise-sensitive; estimates gradient at half-integer positions

### Step 2: Prewitt Operator (1970)

$$G_x^P = \begin{bmatrix} -1 & 0 & 1 \\ -1 & 0 & 1 \\ -1 & 0 & 1 \end{bmatrix} = \begin{bmatrix} 1 \\ 1 \\ 1 \end{bmatrix} \begin{bmatrix} -1 & 0 & 1 \end{bmatrix}$$

**Smoothing vector:** $[1, 1, 1]^T$ (box filter — uniform averaging)

**Derivation:** The Prewitt operator is the simplest separable 3×3 gradient estimator: box smoothing in $y$ combined with central difference in $x$.

### Step 3: Numerical Comparison of Gradient Accuracy

For a diagonal edge $I(x,y) = x + y$ (45° edge, gradient = $(\sqrt{2}/2, \sqrt{2}/2)$):

**True gradient magnitude:** $|\nabla I| = \sqrt{2} \approx 1.414$

**Roberts:** $|K_1 * I| = |1 \cdot (x+y) + (-1)(x+1+y+1)| = 2$, $|K_2| = 0$. Magnitude $= 2$ (error: 41%)

**Prewitt:** $|G_x^P * I| = 3$, $|G_y^P * I| = 3$. Magnitude $= 3\sqrt{2} \approx 4.24$ (correct ratio, but scale factor 3)

**Sobel:** $|G_x^S * I| = 4$, $|G_y^S * I| = 4$. Magnitude $= 4\sqrt{2} \approx 5.66$ (correct ratio, scale factor 4)

**Scharr:** $|G_x^{Sch} * I| = 16$, $|G_y^{Sch} * I| = 16$. Magnitude $= 16\sqrt{2} \approx 22.6$ (correct ratio, scale factor 16)

### Step 4: Angular Accuracy Comparison

The key metric is how well each operator estimates the gradient DIRECTION across all angles.

**Maximum angular error** (deviation from true gradient direction):
- Roberts: 10.2°
- Prewitt: 4.6°
- Sobel: 3.2°
- Scharr: 0.4°

**Scharr is 8× more accurate** than Sobel in direction estimation — crucial for applications where edge orientation matters (e.g., feature matching, structure analysis).

### Step 5: RL Agent Kernel Selection

An RL agent processing images at different noise levels should select different operators:
- Clean images (SNR > 30dB): Scharr (best accuracy)
- Moderate noise (20-30dB): Sobel (good accuracy-noise tradeoff)
- High noise (< 20dB): Smooth first (larger Gaussian), then use Sobel/Scharr on the smoothed image

## 1. Image Gradient

### Definition:
The gradient of an image $I$ at point $(x,y)$:

$$\nabla I = \begin{bmatrix} \frac{\partial I}{\partial x} \\ \frac{\partial I}{\partial y} \end{bmatrix} = \begin{bmatrix} G_x \\ G_y \end{bmatrix}$$

### Discrete Approximation:
$$G_x[m,n] \approx I[m, n+1] - I[m, n-1]$$
$$G_y[m,n] \approx I[m+1, n] - I[m-1, n]$$

### Gradient Magnitude:
$$|\nabla I| = \sqrt{G_x^2 + G_y^2}$$

### Gradient Direction:
$$\theta = \arctan\left(\frac{G_y}{G_x}\right)$$

**Edges occur where $|\nabla I|$ is large!**

## 2. Sobel Operator

$$K_x = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix}, \quad K_y = \begin{bmatrix} -1 & -2 & -1 \\ 0 & 0 & 0 \\ 1 & 2 & 1 \end{bmatrix}$$

Sobel = Smoothing (Gaussian) + Differentiation

## 3. Canny Edge Detection

Complete algorithm:
1. **Gaussian smoothing:** $I_s = G_\sigma * I$
2. **Gradient computation:** $G_x = K_x * I_s$, $G_y = K_y * I_s$
3. **Non-maximum suppression:** Keep only local maxima along gradient direction
4. **Double thresholding:** $T_{low}$, $T_{high}$
5. **Hysteresis:** Connect weak edges to strong edges

## Scale Selection in Edge Detection — Automatic σ Determination

The scale parameter $\sigma$ in Gaussian-based edge detectors controls the tradeoff between noise sensitivity and spatial precision. Choosing $\sigma$ automatically is a fundamental problem.

### Step 1: The Scale-Space Framework

At each scale $\sigma$, the image is represented as $L(x, y; \sigma) = I * G_\sigma$. Edges detected at scale $\sigma$ are features that are "visible" at that resolution.

### Step 2: Scale-Normalized Derivatives

**Problem:** As $\sigma$ increases, the gradient magnitude decreases (because smoothing flattens edges):
$$\|\nabla L_\sigma\| \to 0 \text{ as } \sigma \to \infty$$

This makes it impossible to compare edge strengths across scales.

**Solution (Lindeberg, 1998):** Use **scale-normalized** derivatives:
$$\|\nabla L_\sigma\|_{\text{norm}} = \sigma \cdot \|\nabla L_\sigma\|$$

The factor $\sigma$ compensates for the amplitude reduction due to smoothing.

**Proof of compensation:** For an ideal step edge:
$$\frac{\partial L}{\partial x}\bigg|_{\text{edge}} = \frac{1}{\sqrt{2\pi}\sigma}$$

Normalized: $\sigma \cdot \frac{1}{\sqrt{2\pi}\sigma} = \frac{1}{\sqrt{2\pi}}$ — independent of scale! $\blacksquare$

### Step 3: Automatic Scale Selection

**Principle:** The characteristic scale of a feature is the scale at which the normalized gradient response is maximized:

$$\sigma^* = \arg\max_\sigma \sigma \cdot \|\nabla L(x, y; \sigma)\|$$

### Step 4: Multi-Scale Edge Detection

Compute edges at multiple scales $\sigma_1 < \sigma_2 < \cdots < \sigma_K$ and combine:

$$E_{\text{multi}} = \bigcup_{k=1}^{K} \{(x, y) : (x,y) \text{ is Canny edge at scale } \sigma_k \text{ AND } \sigma_k = \sigma^*(x,y)\}$$

Each edge is detected at its **natural** scale — fine edges at small $\sigma$, coarse edges at large $\sigma$.

### Step 5: Connection to Wavelet Edge Detection

The wavelet transform at scale $s$ using a wavelet $\psi = G'_\sigma$:

$$W_s f(x) = f * \psi_s(x) = f * (s \cdot G'_{s\sigma})(x) = s \cdot \frac{\partial}{\partial x}(f * G_{s\sigma})$$

This IS the scale-normalized gradient — wavelets naturally implement multi-scale edge detection.

### Step 6: RL Agent for Scale Selection

An RL agent processing images at multiple scales can learn:
- **State:** Multi-scale gradient responses $[\|\nabla L_{\sigma_1}\|, \ldots, \|\nabla L_{\sigma_K}\|]$
- **Action:** Select optimal $\sigma$ for each image region
- **Reward:** Edge detection F1-score against ground truth

The agent outperforms fixed-scale edge detection by adapting $\sigma$ to the local noise level and edge characteristics.

## Edge Detection in Color Images — Gradient Generalization to Vector Fields

### Step 1: The Multi-Channel Gradient Problem

For a color image $\mathbf{I}(x,y) = (R(x,y), G(x,y), B(x,y))^T$, each channel has its own gradient:

$$\nabla R = (R_x, R_y), \quad \nabla G = (G_x, G_y), \quad \nabla B = (B_x, B_y)$$

**Naive approach:** Compute gradient magnitude per channel and take the maximum:
$$|\nabla I|_{\text{naive}} = \max(|\nabla R|, |\nabla G|, |\nabla B|)$$

**Problem:** This misses edges that are visible only across channels (e.g., a red-green boundary where luminance is constant).

### Step 2: Di Zenzo's Color Edge Detector (1986)

**Construct the Jacobian matrix:**
$$\mathbf{J} = \begin{bmatrix} R_x & R_y \\ G_x & G_y \\ B_x & B_y \end{bmatrix} \in \mathbb{R}^{3 \times 2}$$

**Form the structure tensor:**
$$\mathbf{S} = \mathbf{J}^T\mathbf{J} = \begin{bmatrix} R_x^2 + G_x^2 + B_x^2 & R_xR_y + G_xG_y + B_xB_y \\ R_xR_y + G_xG_y + B_xB_y & R_y^2 + G_y^2 + B_y^2 \end{bmatrix} = \begin{bmatrix} g_{xx} & g_{xy} \\ g_{xy} & g_{yy} \end{bmatrix}$$

### Step 3: Edge Magnitude and Direction from Eigenvalues

The eigenvalues of $\mathbf{S}$ give the maximum and minimum rates of color change:

$$\lambda_{\pm} = \frac{(g_{xx} + g_{yy}) \pm \sqrt{(g_{xx} - g_{yy})^2 + 4g_{xy}^2}}{2}$$

**Edge magnitude:** $|\nabla \mathbf{I}| = \sqrt{\lambda_+}$

**Edge direction:** $\theta = \frac{1}{2}\arctan\frac{2g_{xy}}{g_{xx} - g_{yy}}$

### Step 4: Proof That This Captures All Color Edges

**Theorem:** $\lambda_+ = 0$ iff the image is locally constant in ALL channels simultaneously.

**Proof:** $\lambda_+ = 0$ iff both eigenvalues are zero iff $\mathbf{S} = \mathbf{0}$ iff $\mathbf{J}^T\mathbf{J} = \mathbf{0}$ iff $\mathbf{J} = \mathbf{0}$ (since $\|\mathbf{Jv}\|^2 = \mathbf{v}^T\mathbf{J}^T\mathbf{J}\mathbf{v} = 0$ for all $\mathbf{v}$) iff $R_x = R_y = G_x = G_y = B_x = B_y = 0$. $\blacksquare$

**Comparison:** The grayscale gradient magnitude is $\sqrt{g_{xx}}$ (only measures horizontal gradient). The color edge magnitude $\sqrt{\lambda_+}$ is always $\geq$ the grayscale magnitude, detecting edges that are invisible in any single channel.

### RL Impact

For RL agents that use edge features (e.g., for navigation or object detection), color edge detection provides strictly more information than grayscale edges. The additional computation ($3 \times$ gradient + eigenvalue) is modest and pays off in edge detection accuracy.

In [ ]:
# Create a sample image with clear edges
size = 128
img = np.zeros((size, size), dtype=np.float64)

# Rectangle
img[30:100, 30:100] = 200
# Circle
for i in range(size):
    for j in range(size):
        if (i-64)**2 + (j-64)**2 < 20**2:
            img[i, j] = 100
# Triangle
for i in range(20):
    img[10+i, 64-i:64+i] = 150

# Add slight noise
img_noisy = img + np.random.normal(0, 5, img.shape)
img_uint8 = np.clip(img_noisy, 0, 255).astype(np.uint8)

# Apply edge detection methods
# Sobel
sobel_x = cv2.Sobel(img_uint8, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(img_uint8, cv2.CV_64F, 0, 1, ksize=3)
sobel_mag = np.sqrt(sobel_x**2 + sobel_y**2)
sobel_dir = np.arctan2(sobel_y, sobel_x)

# Laplacian
laplacian = cv2.Laplacian(img_uint8, cv2.CV_64F)

# Canny
canny = cv2.Canny(img_uint8, 50, 150)

# Visualize
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

axes[0, 0].imshow(img_uint8, cmap='gray')
axes[0, 0].set_title('Original Image')

axes[0, 1].imshow(sobel_x, cmap='RdBu')
axes[0, 1].set_title('Sobel Gx\n(Horizontal edges)')

axes[0, 2].imshow(sobel_y, cmap='RdBu')
axes[0, 2].set_title('Sobel Gy\n(Vertical edges)')

axes[0, 3].imshow(sobel_mag, cmap='hot')
axes[0, 3].set_title('|∇I| = √(Gx² + Gy²)\n(Edge magnitude)')

axes[1, 0].imshow(sobel_dir, cmap='hsv')
axes[1, 0].set_title('θ = arctan(Gy/Gx)\n(Edge direction)')

axes[1, 1].imshow(np.abs(laplacian), cmap='hot')
axes[1, 1].set_title('|Laplacian|\n(∇²I = ∂²I/∂x² + ∂²I/∂y²)')

axes[1, 2].imshow(canny, cmap='gray')
axes[1, 2].set_title('Canny Edges\n(Complete algorithm)')

axes[1, 3].text(0.5, 0.5,
    'RL Connection:\n\n'
    'Edge maps tell the agent\n'
    'WHERE objects are.\n\n'
    'Edge strength = confidence\n'
    'that boundary exists.\n\n'
    'Reward: preserve edges\n'
    'while removing noise!',
    ha='center', va='center', fontsize=11,
    bbox=dict(boxstyle='round', facecolor='lightyellow'),
    transform=axes[1, 3].transAxes)
axes[1, 3].axis('off')

for ax in axes.flat:
    if ax != axes[1, 3]:
        ax.axis('off')

plt.suptitle('Edge Detection Methods — From Gradient to Canny', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('edge_detection.png', dpi=150, bbox_inches='tight')
plt.show()

## Canny Edge Detector — Optimality Criteria and Full Derivation

John Canny (1986) derived his edge detector by posing edge detection as an optimization problem. The result is the provably optimal edge detector under three specific criteria.

### Step 1: Canny's Three Optimality Criteria

**Criterion 1 — Good Detection (low miss rate):**
The probability of failing to detect a true edge should be low, and the probability of falsely marking a non-edge should be low.

Mathematically, maximize the signal-to-noise ratio:
$$\text{SNR} = \frac{\left|\int_{-\infty}^{0} f(x) \, dx\right|}{\sigma \sqrt{\int_{-\infty}^{0} f^2(x) \, dx}}$$

where $f(x)$ is the filter impulse response and $\sigma$ is the noise standard deviation.

**Criterion 2 — Good Localization (precise position):**
The detected edge should be as close as possible to the true edge position.

$$\text{Localization} = \frac{|f'(0)|}{\sigma \sqrt{\int_{-\infty}^{0} f'^2(x) \, dx}}$$

**Criterion 3 — Single Response (one detection per edge):**
Each true edge should produce only one detection, not multiple nearby responses.

The mean distance between adjacent zero crossings of $f'(x)$ applied to noise:
$$x_{\text{zc}} = \pi \sqrt{\frac{\int f'^2(x) \, dx}{\int f''^2(x) \, dx}}$$

### Step 2: Canny's Optimization Problem

Maximize: $\text{SNR} \times \text{Localization}$

Subject to: the single-response constraint $x_{\text{zc}} = $ const.

Using the calculus of variations, Canny showed that the optimal filter satisfies the differential equation:
$$2f(x) - 2\lambda_1 f''(x) + 2\lambda_2 f'''(x) = 0$$

### Step 3: The Gaussian Approximation

The exact solution is a sum of exponentials, but Canny proved that the **first derivative of a Gaussian** is a near-optimal approximation:

$$f(x) = -\frac{x}{\sigma^2}\exp\left(-\frac{x^2}{2\sigma^2}\right) = -G'(x; \sigma)$$

The optimality ratio (Canny's filter performance / optimal performance) is approximately 0.96 — within 4% of perfect.

### Step 4: Non-Maximum Suppression — Mathematical Formulation

**Goal:** Thin edges to 1-pixel width by keeping only local maxima along the gradient direction.

**Algorithm:** For each pixel $(x, y)$ with gradient magnitude $M(x,y)$ and direction $\theta(x,y)$:

$$M_{\text{thin}}(x, y) = \begin{cases} M(x,y) & \text{if } M(x,y) \geq M(x + \cos\theta, y + \sin\theta) \\ & \quad \text{AND } M(x,y) \geq M(x - \cos\theta, y - \sin\theta) \\ 0 & \text{otherwise} \end{cases}$$

**Proof of correctness:** Along a 1D profile perpendicular to an ideal step edge convolved with a Gaussian, the gradient magnitude has exactly one maximum (at the edge center). NMS preserves this single maximum and suppresses all other points. $\blacksquare$

### Step 5: Hysteresis Thresholding — Connected Component Analysis

**Two thresholds:** $T_{\text{low}} < T_{\text{high}}$

1. **Strong edges:** $\{(x,y) : M_{\text{thin}}(x,y) > T_{\text{high}}\}$ — definitely edges
2. **Weak edges:** $\{(x,y) : T_{\text{low}} < M_{\text{thin}}(x,y) \leq T_{\text{high}}\}$ — edges only if connected to strong edges
3. **Non-edges:** $\{(x,y) : M_{\text{thin}}(x,y) \leq T_{\text{low}}\}$ — definitely not edges

**Connected component formulation:** A weak edge pixel is retained if and only if there exists a path of edge pixels (weak or strong) connecting it to at least one strong edge pixel in the 8-connected sense.

**Typical ratio:** $T_{\text{high}} / T_{\text{low}} \in [2, 3]$ (Canny recommended 2:1 or 3:1)

## Laplacian of Gaussian — Proof That Zero Crossings Locate Edges

The Laplacian of Gaussian (LoG) is a second-order edge detector that finds edges at zero crossings rather than at maxima.

### Step 1: The Laplacian Operator

The Laplacian is the sum of second partial derivatives:
$$\nabla^2 I = \frac{\partial^2 I}{\partial x^2} + \frac{\partial^2 I}{\partial y^2}$$

### Step 2: Why Second Derivative Finds Edges

**Theorem:** At an ideal step edge (smoothed by observation), the intensity profile $I(x)$ has an inflection point. The second derivative is zero at inflection points.

**Proof:** Consider an edge modeled as $I(x) = \frac{1}{2}\left[1 + \text{erf}\left(\frac{x}{\sqrt{2}\sigma}\right)\right]$ (Gaussian-smoothed step).

$$I'(x) = \frac{1}{\sqrt{2\pi}\sigma}e^{-x^2/2\sigma^2} \quad \text{(Gaussian — has maximum at } x = 0\text{)}$$

$$I''(x) = -\frac{x}{\sqrt{2\pi}\sigma^3}e^{-x^2/2\sigma^2}$$

$I''(0) = 0$: the second derivative crosses zero exactly at the edge location. $\blacksquare$

### Step 3: LoG Derivation

The Laplacian of a Gaussian-smoothed image:
$$\nabla^2(G_\sigma * I) = (\nabla^2 G_\sigma) * I$$

by linearity of differentiation and convolution. The LoG kernel is:

$$\nabla^2 G(x,y;\sigma) = \frac{\partial^2 G}{\partial x^2} + \frac{\partial^2 G}{\partial y^2}$$

**Computing $\frac{\partial^2 G}{\partial x^2}$:**

$$\frac{\partial G}{\partial x} = -\frac{x}{\sigma^2}G(x,y;\sigma)$$

$$\frac{\partial^2 G}{\partial x^2} = -\frac{1}{\sigma^2}G + \frac{x^2}{\sigma^4}G = \frac{x^2 - \sigma^2}{\sigma^4}G$$

Similarly: $\frac{\partial^2 G}{\partial y^2} = \frac{y^2 - \sigma^2}{\sigma^4}G$

$$\therefore \nabla^2 G = \frac{x^2 + y^2 - 2\sigma^2}{\sigma^4} \cdot G(x,y;\sigma) = \frac{r^2 - 2\sigma^2}{2\pi\sigma^6}\exp\left(-\frac{r^2}{2\sigma^2}\right)$$

where $r^2 = x^2 + y^2$. $\blacksquare$

### Step 4: The "Mexican Hat" Shape

The LoG has a characteristic shape: negative at center, positive ring, tails to zero:
- $\nabla^2 G = 0$ at $r = \sigma\sqrt{2}$ (the zero-crossing radius)
- Negative for $r < \sigma\sqrt{2}$ (center excitation)
- Positive for $r > \sigma\sqrt{2}$ (surround inhibition)

This center-surround pattern matches the receptive fields of retinal ganglion cells, suggesting the visual system implements LoG-like edge detection.

### Step 5: Difference of Gaussians (DoG) Approximation

**Theorem:** $\nabla^2 G_\sigma \approx \frac{G_{\sigma_1} - G_{\sigma_2}}{\sigma_1^2 - \sigma_2^2}$ where $\sigma_2 \approx 1.6\sigma_1$

**Proof:** For small $\Delta\sigma$, by the definition of derivative:
$$\frac{\partial G}{\partial \sigma} \approx \frac{G_{\sigma+\Delta\sigma} - G_\sigma}{\Delta\sigma}$$

It can be shown that $\sigma\frac{\partial G}{\partial \sigma} = \sigma^2 \nabla^2 G$ (the diffusion equation). Therefore:
$$\nabla^2 G_\sigma \approx \frac{G_{\sigma_2} - G_{\sigma_1}}{\sigma_2^2 - \sigma_1^2} \quad \blacksquare$$

The DoG is computationally cheaper (two Gaussian blurs vs. computing second derivatives) and is used in SIFT feature detection.

## Non-Maximum Suppression — Subpixel Edge Localization

Non-maximum suppression (NMS) is the critical step that converts a thick gradient response into thin, one-pixel-wide edges. Here we derive the mathematical formulation and prove its correctness.

### Step 1: The Problem

After computing the gradient magnitude $M(x, y)$ and direction $\theta(x, y)$, many pixels near an edge have large $M$ values. We need to select only the single pixel at the exact edge location.

### Step 2: Mathematical Formulation

**For each pixel $(x, y)$:**
1. Round $\theta(x, y)$ to one of 4 directions: $0°, 45°, 90°, 135°$
2. Compare $M(x, y)$ with its two neighbors along the gradient direction
3. Suppress (set to 0) if not a local maximum

**Formally, let $\mathbf{n} = (\cos\theta, \sin\theta)$ be the unit gradient direction:**

$$M_{\text{NMS}}(x, y) = \begin{cases} M(x, y) & \text{if } M(x,y) \geq M(x + n_x, y + n_y) \\ & \quad \text{AND } M(x,y) \geq M(x - n_x, y - n_y) \\ 0 & \text{otherwise} \end{cases}$$

### Step 3: Interpolation for Non-Integer Neighbor Positions

Since $\theta$ is generally not a multiple of $45°$, the neighbors $(x \pm \cos\theta, y \pm \sin\theta)$ fall at non-integer positions. We interpolate using the two nearest grid points.

**Example for $\theta \in (0°, 45°)$:** The gradient points between the east and northeast directions. The neighbor magnitude is interpolated as:

$$M_{\text{interp}} = (1 - \tan\theta) \cdot M(x+1, y) + \tan\theta \cdot M(x+1, y+1)$$

### Step 4: Proof That NMS Produces One-Pixel-Wide Edges

**Theorem:** For an ideal step edge convolved with a Gaussian, NMS preserves exactly one pixel per cross-section of the edge.

**Proof:** Consider a 1D profile $g(t)$ across an edge, where $g(t) = G(t;\sigma) * \text{step}(t)$. The gradient magnitude along this profile is:

$$m(t) = |g'(t)| = \frac{1}{\sqrt{2\pi}\sigma}e^{-t^2/2\sigma^2}$$

This has a single maximum at $t = 0$ (the edge center):
$$m'(t) = -\frac{t}{\sqrt{2\pi}\sigma^3}e^{-t^2/2\sigma^2} = 0 \iff t = 0$$

Since $m''(0) = -\frac{1}{\sqrt{2\pi}\sigma^3} < 0$, this is a strict maximum. NMS preserves only this maximum, giving a one-pixel-wide edge at exactly $t = 0$. $\blacksquare$

### Step 5: Subpixel Refinement

For higher precision, fit a parabola to the three gradient values along the gradient direction:

$$m(t) \approx at^2 + bt + c$$

Using $m(-1), m(0), m(1)$:
$$t_{\text{peak}} = -\frac{b}{2a} = -\frac{m(1) - m(-1)}{2(m(1) - 2m(0) + m(-1))}$$

This gives the edge location to subpixel accuracy — important for precision measurement applications.

### Step 6: Computational Considerations

NMS is inherently sequential along the gradient direction but parallel across different gradient directions. In an RL context, where the agent processes many frames:
- GPU-friendly: gradient computation is parallel (convolution)
- NMS can be implemented as a conditional max-pooling operation
- Total cost: $O(N^2)$ — linear in image size, dominated by gradient computation

## Edge Linking and the Hough Transform — From Local to Global Edge Analysis

### Step 1: The Gap Problem

Even with optimal Canny detection, edge maps often have gaps due to:
- Noise disrupting weak edges
- Gradual transitions at some edge segments
- Occlusions and transparency

Edge linking fills these gaps by using global geometric constraints.

### Step 2: Hough Transform — Detecting Lines in Edge Maps

**Key insight:** A line in image space corresponds to a point in parameter space, and vice versa.

**Line parametrization (normal form):** $x\cos\theta + y\sin\theta = \rho$

where $\rho$ is the perpendicular distance from the origin and $\theta$ is the angle of the normal.

**Transform:** Each edge pixel $(x_i, y_i)$ maps to a sinusoidal curve in $(\rho, \theta)$ space:
$$\rho = x_i\cos\theta + y_i\sin\theta$$

### Step 3: Voting and Accumulator

1. Discretize $(\rho, \theta)$ space into an accumulator array $A[\rho, \theta]$
2. For each edge pixel, increment all cells on its sinusoidal curve
3. Peaks in $A$ correspond to lines in the image

**Proof of correctness:** If $n$ collinear edge pixels lie on the line $(\rho_0, \theta_0)$, all their sinusoidal curves pass through $A[\rho_0, \theta_0]$. Therefore $A[\rho_0, \theta_0] = n$, creating a peak. $\blacksquare$

**Complexity:** For $N$ edge pixels and an accumulator of size $P \times Q$: $O(NQ)$ (each pixel votes in $Q$ $\theta$-bins).

### Step 4: Generalized Hough Transform

For arbitrary shapes defined by a template table $R$:

For each edge pixel $(x, y)$ with gradient direction $\phi$:
1. Look up all possible reference point offsets $(dx, dy) \in R(\phi)$
2. Vote: $A[x - dx, y - dy] \mathrel{+}= 1$

Peaks in $A$ indicate locations where the template shape is present.

**Proof of generality:** Any shape can be described by its boundary gradient directions and their distances to a reference point. The voting process is equivalent to template matching via gradient features. $\blacksquare$

### Step 5: Probabilistic Hough Transform

The standard Hough transform is $O(NQ)$ — expensive for large images. The probabilistic variant (Matas et al., 2000) randomly samples a subset of edge pixels:

1. Randomly select an edge pixel
2. Randomly select a $\theta$ direction
3. Walk along the line at $\theta$ through the pixel
4. If enough edge pixels are found along this line, declare it a line segment
5. Remove these pixels from the edge map

**Complexity:** $O(N)$ expected time — much faster than standard Hough.

### Step 6: RL Connection — Edge-Based Feature Extraction

An RL agent for autonomous navigation can use Hough-detected lines as high-level features:
- Lane markings → detected as line pairs
- Building edges → dominant vertical/horizontal lines
- Vanishing points → intersections of detected lines

This converts the raw edge map into a compact geometric description of the scene, serving as a structured state representation for the agent's policy.

---
**Next:** Module 2.3 — Morphological Operations